In [ ]:
!pip install requests beautifulsoup4

In [ ]:
# 巨潮网 科创板 招股说明书爬虫（修复版）
# 修复点：
#   1. 读取 totalpages 控制翻页边界，避免越界回卷
#   2. 用 set 对 secCode 去重，避免同一公司被重复计数
#   3. download_pdf 区分"已存在"和"新下载"，避免把跳过当成功
#   4. column 改为 szse（巨潮统一查询接口的正确值）
#   5. 用 Session 复用连接

import requests
import time
import random
import os
from tqdm import tqdm

SAVE_DIR = "科创板招股书"
os.makedirs(SAVE_DIR, exist_ok=True)

BASE_URL = "https://www.cninfo.com.cn/new/hisAnnouncement/query"
PDF_BASE = "https://static.cninfo.com.cn/"

MAX_DOWNLOAD = 50

UA_LIST = [
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 Chrome/128.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/605.1.15 Safari/605.1.15",
    "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 Chrome/120.0.0.0 Safari/537.36",
]

HEADERS = {
    "Accept": "application/json, text/javascript, */*; q=0.01",
    "Accept-Language": "zh-CN,zh;q=0.9",
    "Content-Type": "application/x-www-form-urlencoded; charset=UTF-8",
    "Referer": "https://www.cninfo.com.cn/",
    "X-Requested-With": "XMLHttpRequest",
}

session = requests.Session()


def get_announcement_page(page_num):
    headers = HEADERS.copy()
    headers["User-Agent"] = random.choice(UA_LIST)
    data = {
        "pageNum": page_num,
        "pageSize": 50,
        "tabName": "fulltext",
        "column": "szse",
        "plate": "kcb",
        "stock": "",
        "searchkey": "首次公开发行股票并在科创板上市招股说明书",
        "category": "",
        "seDate": "2019-07-22~2026-12-31",
        "sortName": "time",
        "sortType": "desc",
        "isHLtitle": "false",
    }
    try:
        resp = session.post(BASE_URL, headers=headers, data=data, timeout=20)
        resp.raise_for_status()
        return resp.json()
    except Exception as e:
        print(f"第{page_num}页请求失败：{e}")
        return None


def download_pdf(pdf_url, sec_code, sec_name):
    """返回值：'new' = 新下载成功，'exists' = 已存在，'fail' = 失败"""
    filename = f"{sec_code}_{sec_name}_科创板招股说明书.pdf"
    filename = filename.replace("/", "").replace("\\", "")
    save_path = os.path.join(SAVE_DIR, filename)

    if os.path.exists(save_path):
        print(f"⏭️  已存在，跳过：{filename}")
        return "exists"

    try:
        resp = session.get(pdf_url, stream=True, timeout=30)
        resp.raise_for_status()
        total_size = int(resp.headers.get("content-length", 0))
        with open(save_path, "wb") as f, tqdm(
            total=total_size, unit="B", unit_scale=True, desc=sec_code
        ) as bar:
            for chunk in resp.iter_content(chunk_size=8192):
                f.write(chunk)
                bar.update(len(chunk))
        print(f"✅ 下载成功：{filename}")
        return "new"
    except Exception as e:
        print(f"❌ 下载失败：{e}")
        return "fail"


def crawl_kcb():
    print("🚀 开始爬取巨潮网科创板招股说明书\n")

    # 先取一次拿 totalpages
    first = get_announcement_page(1)
    if not first:
        print("首页请求失败，退出")
        return
    total_pages = first.get("totalpages", 1) or 1
    total_records = first.get("totalRecordNum", 0)
    print(f"📄 接口返回 totalpages={total_pages}, totalRecordNum={total_records}\n")

    downloaded_codes = set()  # 已下载（含已存在）的 secCode
    seen_codes = set()        # 已遇到（命中筛选）的 secCode，用于去重

    page = 1
    while page <= total_pages and len(downloaded_codes) < MAX_DOWNLOAD:
        print(f"\n===== 第 {page}/{total_pages} 页 =====")
        data = first if page == 1 else get_announcement_page(page)
        if not data or not data.get("announcements"):
            print("无数据，提前结束")
            break

        for ann in data["announcements"]:
            title = ann["announcementTitle"]
            sec_code = ann["secCode"]
            sec_name = ann["secName"]
            url = ann.get("adjunctUrl")

            # 板块过滤：688 开头才是科创板（巨潮接口 plate=kcb 实测对全文检索无效）
            if not sec_code.startswith("688"):
                continue
            # 标题筛选：要正式招股书，排除摘要/申报稿/提示性公告等
            if "招股说明书" not in title:
                continue
            if any(x in title for x in ["摘要", "申报稿", "上会稿", "修正", "问询", "提示", "意向书", "刊发H股"]):
                continue
            if not url:
                continue

            # secCode 去重：同一公司只处理一次
            if sec_code in seen_codes:
                continue
            seen_codes.add(sec_code)

            if len(downloaded_codes) >= MAX_DOWNLOAD:
                break

            result = download_pdf(PDF_BASE + url, sec_code, sec_name)
            if result in ("new", "exists"):
                downloaded_codes.add(sec_code)
                print(f"📊 进度 {len(downloaded_codes)}/{MAX_DOWNLOAD}")
                if result == "new":
                    time.sleep(random.uniform(2, 4))

        page += 1
        time.sleep(1)

    print(f"\n🎉 任务完成！共下载/确认 {len(downloaded_codes)} 家科创板招股说明书")
    print(f"   命中标题筛选的 secCode 总数：{len(seen_codes)}")


if __name__ == "__main__":
    crawl_kcb()
